In [145]:
!pip install gym

In [146]:
!pip install sklearn

In [147]:
import random
import numpy as np
import networkx as nx
import gymnasium as gym
from gymnasium import spaces
from sklearn.preprocessing import LabelEncoder

## Define the Semantic Knowledge Graph

Create a class that is able to represent a semantic knowledge graph. The graph should be directed and should allow adding concepts and relationships between them. The graph should also allow querying properties such as neighbors, learned status, and distances.


In [148]:
class SemanticKnowledgeGraph:
    """A class representing a semantic knowledge graph.

    This graph is implemented as a directed graph using NetworkX. It allows
    adding concepts, defining relationships between them, and querying
    properties such as neighbors, learned status, and distances.

    Attributes:
        graph (networkx.DiGraph): The directed graph representing the knowledge
            graph.
    """
    def __init__(self) -> None:
        """Initializes the semantic knowledge graph."""
        self.graph = nx.DiGraph()

    def add_concept(self, concept: str) -> None:
        """Adds a concept to the graph.

        Args:
            concept (str): The concept to add.
        """
        self.graph.add_node(concept, learned=False, accumulated_reward=0)

    def add_relationship(self, concept1: str, concept2: str) -> None:
        """Adds a directed relationship between two concepts.

        Args:
            concept1 (str): The first concept.
            concept2 (str): The second concept.
        """
        self.graph.add_edge(concept1, concept2)

    def get_neighbors(self, concept: str) -> list[str]:
        """Gets the neighbors of a concept.

        Args:
            concept (str): The concept to query.

        Returns:
            list: A list of neighboring concepts.
        """
        return list(self.graph.successors(concept))

    def mark_learned(self, concept: str) -> None:
        """Marks a concept as learned.

        Args:
            concept (str): The concept to mark as learned.
        """
        self.graph.nodes[concept]['learned'] = True

    def is_learned(self, concept: str) -> bool:
        """Checks if a concept is learned.

        Args:
            concept (str): The concept to check.

        Returns:
            bool: True if the concept is learned, False otherwise.
        """
        return self.graph.nodes[concept].get('learned', False)

    def get_distance(self, start: str, end: str) -> float:
        """Calculates the shortest path distance between two concepts.

        If no directed path exists, it attempts to calculate the distance
        using an undirected version of the graph.

        Args:
            start (str): The starting concept.
            end (str): The target concept.

        Returns:
            float: The shortest path distance, or infinity if no path
            exists.
        """
        try:
            return nx.shortest_path_length(self.graph, source=start, target=end)
        except nx.NetworkXNoPath:
            # Use undirected shortest path if no directed path exists
            undirected_graph = self.graph.to_undirected()
            try:
                return nx.shortest_path_length(undirected_graph, source=start, target=end)
            except nx.NetworkXNoPath:
                return float('inf')

## Define the Student

The student represents the simulated learner in the environment. Its goal is to learn all concepts that are defined via the knowledge graph.
It has a knowledge state that keeps track of the concepts learned. The student can query the graph to determine the reward for learning a new concept based on its distance from the current concept and its talent distribution.

In [149]:
# Define the Student
class Student:
    """Represents a student learning concepts in a graph-based environment.

    Attributes:
        knowledge_state (set): The set of concepts the student has learned.
        talent_distribution (dict): Maps concepts to their difficulty levels.
        graph (nx.DiGraph): The graph representing the relationships between concepts.
        rewards (list): A list of rewards accumulated during learning.

    Methods:
        query(current_concept, concept):
            Calculates the reward for attempting to learn a concept based on
            distance and talent distribution.

        learn(concept):
            Adds a concept to the knowledge state if sufficient reward is
            accumulated.
    """
    def __init__(self, talent_distribution: dict, graph: nx.DiGraph) -> None:
        self.knowledge_state = set()
        self.talent_distribution = talent_distribution  # Dict mapping concept to difficulty
        self.graph = graph
        self.rewards = []

    def query(self, current_concept: str, concept: str) -> float:
        """Calculates the reward for attempting to learn a concept.

        Args:
            current_concept (str): The current concept the student is focused on.
            concept (str): The target concept to learn.

        Returns:
            float: The reward for attempting to learn the concept.
        """
        if concept in self.knowledge_state:
            return 0  # Already learned
        distance = self.graph.get_distance(current_concept, concept)
        distance_factor = max(0.1, 1 / (distance + 1))  # Closer nodes have higher rewards
        reward = np.random.beta(2, self.talent_distribution.get(concept, 2)) * distance_factor  # Talent + distance
        self.graph.graph.nodes[concept]['accumulated_reward'] += reward
        self.rewards.append(reward)
        return reward

    def learn(self, concept: str) -> None:
        """Adds a concept to the knowledge state if sufficient reward is accumulated.

        Args:
            concept (str): The concept to be learned.
        """
        if concept not in self.knowledge_state:
            if self.graph.graph.nodes[concept]['accumulated_reward'] >= 1:  # Accumulate enough reward to learn
                self.knowledge_state.add(concept)
                self.graph.mark_learned(concept)

## Learning Environment

The learning environment is defined using the OpenAI Gym framework. It simulates the interaction between the student and the knowledge graph. The environment allows the student to take actions (learn concepts) and receive rewards based on their learning progress.

> **Note:** The environment is designed to be compatible with reinforcement learning algorithms, allowing for the integration of various RL agents.

The environment is initialized with a graph and a student. The `reset` method initializes the environment, while the `step` method executes an action and returns the new state, reward, and whether the learning process is complete.

> **Attention:** The `current_concept` method currently uses a random state for demonstration purposes. This should be fixed.



In [150]:
# Define the RL Environment
class LearningEnv(gym.Env):
    """Gym environment for learning concepts in a knowledge graph.

    This environment models a learning process where a student interacts
    with a graph of concepts. The student learns by transitioning between
    concepts and receiving rewards based on their learning progress.

    Attributes:
        graph (nx.DiGraph): A graph representing the relationships between concepts.
        student (Student): An agent that learns concepts from the graph.
        __current_concept (str): The concept the student is currently learning.
        action_dim (int): The number of possible actions (concepts).
        state_dim (int): The number of possible states (concepts).
        action_space (spaces.Discrete): The set of possible actions (concepts).
        observation_space (spaces.Discrete): The set of possible observations.
    """
    def __init__(self, graph: nx.DiGraph, student: Student) -> None:
        """Initializes the LearningEnv with a graph and a student.

        Args:
            graph (Graph): The graph of concepts.
            student (Student): The learning agent.
        """

        super(LearningEnv, self).__init__()

        self.graph = graph

        self.student = student
        self.action_dim = len(self.graph.graph.nodes)
        self.state_dim = len(self.graph.graph.nodes)

        self.__current_concept =  np.random.choice(list(self.graph.graph.nodes))
        self.action_space = spaces.Discrete(len(self.graph.graph.nodes))

        # 5eed an encode that maps nodes to integers for Learner
        self.state_encoder = LabelEncoder()
        print(self.graph.graph.nodes)
        self.state_encoder.fit(list(self.graph.graph.nodes))

    @property
    def current_concept(self) -> int:
        return self.state_encoder.transform([self.__current_concept])[0]

    @property
    def neighbors(self) -> list[int]:
        neighbors = self.state_encoder.transform(self.graph.get_neighbors(self.__current_concept))

    def reset(self, seed: int = None, options: dict =None) -> tuple[int, dict]:
        """Resets the environment to its initial state.

        Args:
            seed (int, optional): A seed for random number generation.
            options (dict, optional): Additional options for resetting.

        Returns:
            tuple: Initial state and an empty dictionary.
        """
        super().reset(seed=seed)
        return self.state_encoder.transform([np.random.choice(list(self.graph.graph.nodes))]), {}

    def step(self, action: int) -> tuple[int, int, float, bool, bool, dict]:
        """Executes a step in the environment based on the given action.

        Args:
            action (int): The index of the next concept to transition to.

        Returns:
            tuple: A tuple containing:
                - action (int): The action taken as it is the new state.
                - reward (float): The reward received for the action.
                - done (bool): Whether the learning process is complete.
                - False (bool): Placeholder for compatibility.
                - dict: An empty dictionary for additional info.
        """
        print("executing step with ", action)
        next_concept = self.state_encoder.inverse_transform(action)[0]
        print("next_concept", next_concept)
        print("current concept", self.__current_concept)
        reward = self.student.query(self.__current_concept, next_concept)
        self.student.learn(next_concept)
        self.__current_concept = next_concept
        done = all(self.graph.is_learned(c) for c in self.graph.graph.nodes)
        return action, reward, done, False, {}

## Learning Agent

Agent that learns to coordinate the learning process

In [151]:
class DynaQAgent:
    """DynaQAgent.

    A Dyna-Q agent for reinforcement learning that combines Q-learning
    with planning.

    Attributes:
    ----------
    n_states (int): Number of states in the environment.
    n_actions (int): Number of possible actions in the environment.
    epsilon (float): Probability of choosing a random action (exploration).
    alpha (float): Learning rate for updating Q-values.
    gamma (float): Discount factor for future rewards.
    planning_steps (int): Number of planning steps to perform.
    q_table (ndarray): Q-values table for state-action pairs.
    model (dict): Model to store state transitions and rewards.
    """

    def __init__(
        self,
        n_states: int,
        n_actions: int,
        epsilon: float = 0.1,
        alpha: float = 0.1,
        gamma: float = 0.95,
        planning_steps: int = 5,
    ) -> None:
        """Initialize the DynaQAgent with the given parameters.

        Args:
        n_states (int): Number of states in the environment.
        n_actions (int): Number of possible actions in the environment.
        epsilon (float): Probability of choosing a random action (exploration).
        alpha (float): Learning rate for updating Q-values.
        gamma (float): Discount factor for future rewards.
        planning_steps (int): Number of planning steps to perform.

        Returns:
        None
        """
        self.n_states = n_states
        self.n_actions = n_actions
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps

        # Initialize Q-table and model
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}

    def predict(self, state: int) -> int:
        """Choose an action.

        Choose an action based on the current state using an epsilon-greedy
        policy.

        Args:
        state (int): The current state of the environment.

        Returns:
        int: The action chosen, either randomly (exploration) or based on the
        highest Q-value (exploitation).
        """
        if random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.n_actions)
        return np.argmax(self.q_table[state])

    def update(self, state: int, action: int, reward: float, next_state: int) -> None:
        """Update Model.

        Update the Q-table and model with the given transition and perform
        planning steps.

        Args:
        state (int): The current state of the environment.
        action (int): The action taken from the current state.
        reward (float): The reward received after taking the action.
        next_state (int): The state transitioned to after taking the action.

        Returns:
        None
        """
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error

        # Store the transition in the model
        self.model[(state, action)] = (reward, next_state)

        # Planning phase
        for _ in range(self.planning_steps):
            s, a = random.choice(list(self.model.keys()))
            r, s_next = self.model[(s, a)]
            best_next_a = np.argmax(self.q_table[s_next])
            td_target = r + self.gamma * self.q_table[s_next][best_next_a]
            td_error = td_target - self.q_table[s][a]
            self.q_table[s][a] += self.alpha * td_error

## Coordinator

In [152]:
class Coordinator:
    """Coordinates the interaction between the student and the environment.

    This class manages the strategy for teaching the student, either using
    reinforcement learning (RL) or a predefined strategy. It also handles
    the agent's training and decision-making process.

    Attributes:
        student: The student object being trained.
        graph: The graph structure representing the knowledge domain.
        use_rl_strategy: A boolean indicating whether to use RL-based strategy.
        agent: The RL agent used for decision-making.
    """
    def __init__(self, student, graph, env):
        """Initializes the Coordinator with a student and a graph.

        Args:
            student: The student object being trained.
            graph: The graph structure representing the knowledge domain.
        """
        self.student = student
        self.graph = graph
        self.use_rl_strategy = False
        # Initialize the agent with the environment's state and action space
        self.agent = DynaQAgent(env.state_dim,env.action_dim)  

    #def train_agent(self, env):
     #   if self.agent is None:
            #self.agent = PPO('MlpPolicy', env, verbose=1)
      #      self.agent.learn(total_timesteps=1) # total_times is the number of steps to train the agent

    def switch_strategy(self):
        """Switches the teaching strategy based on the student's progress.

        If the student's recent progress is below a threshold, switches to
        an RL-based strategy.
        """
        if (len(self.student.rewards)>10):
            print (np.mean(self.student.rewards[-10:]))
            progress = np.mean(self.student.rewards[-10:])

            self.use_rl_strategy = progress < 0.05  # If progress is low, switch to RL-based strategy

    def give_data_to_agent(self, state, action, reward, next_state, done):
        """Provides experience data to the RL agent and trains it.

        Args:
            state: The current state of the environment.
            action: The action taken by the agent.
            reward: The reward received after taking the action.
            next_state: The state of the environment after the action.
            done: A boolean indicating if the episode has ended.
        """
        print("Giving data to agent:", state, action, reward, next_state)
        #self.agent.update(state, action, reward, next_state)

    def query_next_concept(self, env) -> int:
        """Determines the next concept to teach based on the current strategy.

        Args:
            env: The environment object representing the teaching context.

        Returns:
            The next concept to teach.
        """
        self.switch_strategy()
        if self.use_rl_strategy:
            #if self.agent is None:
                #self.train_agent(env)  # Train the agent if it hasn't been trained yet
            action, _ = self.agent.predict(env.action_space.sample())
        else:
            #Student has own strategy
            neighbors = env.neighbors
            action = np.random.choice(neighbors) if neighbors else np.random.choice(list(self.graph.graph.nodes))
        return action

In [153]:
# Setup the Environment
graph = SemanticKnowledgeGraph()
concepts = ['Math', 'Algebra', 'Calculus', 'Physics', 'Mechanics']
for c in concepts:
    graph.add_concept(c)
graph.add_relationship('Math', 'Algebra')
graph.add_relationship('Algebra', 'Calculus')
graph.add_relationship('Physics', 'Mechanics')
print (graph.graph.nodes)

['Math', 'Algebra', 'Calculus', 'Physics', 'Mechanics']


In [154]:
# This should be different for differently embodied students
talent_distribution = {'Math': 2, 'Algebra': 3, 'Calculus': 5, 'Physics': 4, 'Mechanics': 6}
student = Student(talent_distribution, graph)
env = LearningEnv(graph, student)
coordinator = Coordinator(student, graph, env)

['Math', 'Algebra', 'Calculus', 'Physics', 'Mechanics']


In [155]:
# Train agent while student learns
print(list(env.state_encoder.classes_))
for _ in range(5):
    # Next concept to learn
    next_concept = coordinator.query_next_concept(env)
    action = next_concept
    print("Next concept to learn:", action)
    # Should execute the action in the environment
    # how to get the index of the above action in the list of concepts

    # HACK: This should be the action space of the environment
    action = env.state_encoder.transform([action])
    next_state, reward, done, info, _ = env.step(action)  # Take the action in the environment
    coordinator.give_data_to_agent(env.current_concept, action, reward, next_state, done)

[np.str_('Algebra'), np.str_('Calculus'), np.str_('Math'), np.str_('Mechanics'), np.str_('Physics')]
Next concept to learn: Calculus
executing step with  [1]
next_concept Calculus
current concept Calculus
Giving data to agent: 1 [1] 0.05455788370113185 [1]
Next concept to learn: Algebra
executing step with  [0]
next_concept Algebra
current concept Calculus
Giving data to agent: 0 [0] 0.04862644289811548 [0]
Next concept to learn: Mechanics
executing step with  [3]
next_concept Mechanics
current concept Algebra
Giving data to agent: 3 [3] 0.029948143658926563 [3]
Next concept to learn: Mechanics
executing step with  [3]
next_concept Mechanics
current concept Mechanics
Giving data to agent: 3 [3] 0.26936129983875334 [3]
Next concept to learn: Mechanics
executing step with  [3]
next_concept Mechanics
current concept Mechanics
Giving data to agent: 3 [3] 0.15565586125269612 [3]
